# Geological Surface Accuracy


# 0.1 Carica il workspace

eseguire (play) la cella "CARICA SPAZIO DI LAVORO" solo all'apertura del notebook (altrimenti si ricarica lo spazio di lavoro di default cancellando le modifiche):

- verrà caricato l'ambiente di lavoro e la cartella "cartella_files" dove andranno messi i file GOCAD delle superfici che costituiscono il modello 3D (.ts) e gli shape file che contengono le tracce delle sezioni sismiche

  e e i punti relativi ai pozzi 


In [ ]:
# ### CARICA SPAZIO DI LAVORO

import os
import shutil

# Percorso base
base_path = '/content'
repo_path = os.path.join(base_path, 'GeoSurface_Accuracy')

# Funzione per pulire completamente la directory
def clean_repo_directory():
    try:
        # Rimuovi la directory se esiste
        if os.path.exists(repo_path):
            shutil.rmtree(repo_path)
            print(f"Directory {repo_path} rimossa")
    except Exception as e:
        print(f"Errore nella rimozione della directory: {e}")

# Pulisci la directory
clean_repo_directory()

# Cambia nella directory base
os.chdir(base_path)

# Clona il repository
!git clone https://github.com/BaterHub/GeoSurface_Accuracy.git

# Cambia nella directory del repository
%cd GeoSurface_Accuracy


# 0.2 Carica i file nella cartella "cartella_files"

Trascinare i file* del pacchetto costituente il modello 3D nella cartella "working_files_folder"

*NB andranno caricati i seguenti file:

- horizons.ts (deve contenere tutte le geometrie delle superfici)

- shapefile delle tracce di sezioni geologiche e linee sismiche utilizzate per la costruzione della superficie


# 0.3 Eseguire lo script

- Posizionarsi nella cella "LANCIA LO SCRIPT" e eseguire il RUN con "ctrl + F10" oppure dal menù "Runtime > Run cell and below/Esegui questa cella e quelle sottostanti"
- Al termine del RUN verranno generati gli output e un log_file all'interno della working_files_folder.


# 1. Importa librerie e funzioni


In [ ]:
# ### LANCIA LO SCRIPT

## Importa librerie necessarie
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from pyproj import Proj, transform
from scipy.spatial import cKDTree
from scipy.interpolate import griddata
from sklearn.preprocessing import MinMaxScaler
import os
import re
from pathlib import Path

#############################################################################################
## Importa funzioni
import importlib # modulo per il reload delle funzioni

# Reimporta i moduli originali
import files_utils

# Ricarica forzata di ciascun modulo
importlib.reload(files_utils)

# Reimporta le funzioni dai moduli ricaricati
from files_utils import *
#############################################################################################

# Percorso cartella
folder_name = "working_files_folder"
input_dir = os.path.abspath(folder_name)
output_dir = "output_results"


In [ ]:
# Funzioni per griglia e pesi IDWimport numpy as npfrom scipy.spatial import cKDTreedef distance_matrix(x0, y0, x1, y1):    d0 = np.subtract.outer(x0, x1)    d1 = np.subtract.outer(y0, y1)    return np.hypot(d0, d1)def nearest_distance(points, targets):    tree = cKDTree(targets)    dist, idx = tree.query(points, k=1)    return dist, idxdef build_grid(vertices, spacing=5000):    xs, ys = vertices[:, 0], vertices[:, 1]    min_x, max_x = xs.min(), xs.max()    min_y, max_y = ys.min(), ys.max()    gx = np.arange(min_x, max_x + spacing, spacing)    gy = np.arange(min_y, max_y + spacing, spacing)    GX, GY = np.meshgrid(gx, gy)    grid_points = np.c_[GX.ravel(), GY.ravel()]    return GX, GY, grid_pointsdef extract_points_from_wells(wells_gdf):    xs = wells_gdf.geometry.x.values    ys = wells_gdf.geometry.y.values    return xs, ysdef sample_lines_gdf(lines_gdf, step=2000):    pts_x, pts_y = [], []    for geom in lines_gdf.geometry:        if geom.is_empty:            continue        if geom.geom_type == 'LineString':            geoms = [geom]        else:            geoms = list(geom.geoms)        for g in geoms:            num = max(2, int(max(g.length, step) // step))            for f in np.linspace(0, 1, num):                p = g.interpolate(f, normalized=True)                pts_x.append(p.x)                pts_y.append(p.y)    return np.array(pts_x), np.array(pts_y)def compute_horizontal_weights(grid_points, wells_gdf, sections_gdf, power=2, line_step=2000):    weights_list = []    eps = 1e-6    if wells_gdf is not None and not wells_gdf.empty:        wx, wy = extract_points_from_wells(wells_gdf)        if len(wx) > 0:            dists, _ = nearest_distance(grid_points, np.c_[wx, wy])            w = 1 / np.power(dists + eps, power)            w = w / w.max()            weights_list.append(w)    if sections_gdf is not None and not sections_gdf.empty:        lx, ly = sample_lines_gdf(sections_gdf, step=line_step)        if len(lx) > 0:            dists, _ = nearest_distance(grid_points, np.c_[lx, ly])            w = 1 / np.power(dists + eps, power)            w = w / w.max()            weights_list.append(w * 0.7)    if not weights_list:        return None    weights_stack = np.vstack(weights_list)    return weights_stack.mean(axis=0)

In [ ]:
# ### Impostazione delle cartella di lavoro
working_dir = "working_files_folder"
output_dir = "output_results"
crs = 'EPSG:6708'


In [ ]:
# Main Function ed esecuzione proceduradef main(working_dir=working_dir):    print("Avvio dell'analisi dei dati geologici...")    if not os.path.exists(working_dir):        print(f"La cartella {working_dir} non esiste. Creazione in corso...")        os.makedirs(working_dir)        print(f"Cartella {working_dir} creata. Inserisci i file GOCAD .ts e gli shapefile nella cartella.")        return None    print(f"File presenti nella cartella {working_dir}:")    for file in os.listdir(working_dir):        print(f"  - {file}")    vertices, triangles = process_gocad_file(working_dir)    wells_shp = read_wells_shapefile(working_dir)    sections_shp = read_sections_shapefile(working_dir)    has_vertices = vertices is not None and len(vertices) > 0    has_triangles = triangles is not None and len(triangles) > 0    has_wells = wells_shp is not None and not wells_shp.empty    has_sections = sections_shp is not None and not sections_shp.empty    print(f"Stato dei dati:")    print(f"  - Vertices: {'Presenti' if has_vertices else 'Assenti'} - {len(vertices) if has_vertices else 0} vertici")    print(f"  - Triangoli: {'Presenti' if has_triangles else 'Assenti'} - {len(triangles) if has_triangles else 0} triangoli")    print(f"  - Pozzi: {'Presenti' if has_wells else 'Assenti'}")    print(f"  - Sezioni: {'Presenti' if has_sections else 'Assenti'}")    grid_weights = None    grid_points = None    if has_vertices:        GX, GY, grid_points = build_grid(vertices, spacing=5000)        weights = compute_horizontal_weights(grid_points, wells_shp, sections_shp, power=2, line_step=2000)        if weights is not None:            os.makedirs(output_dir, exist_ok=True)            grid_weights = weights.reshape(GX.shape)            pd.DataFrame({'x': grid_points[:,0], 'y': grid_points[:,1], 'weight': weights}).to_csv(                os.path.join(output_dir, 'horizontal_accuracy_grid.csv'), index=False)            fig_w = plt.figure(figsize=(10, 8))            plt.pcolormesh(GX, GY, grid_weights, cmap='viridis', shading='auto')            plt.colorbar(label='Peso (accuratezza orizzontale)')            if has_wells:                plt.scatter(wells_shp.geometry.x, wells_shp.geometry.y, s=8, color='red', label='Pozzi')            if has_sections:                for geom in sections_shp.geometry:                    if geom is None or geom.is_empty:                        continue                    if geom.geom_type == 'LineString':                        xs, ys = geom.xy                        plt.plot(xs, ys, color='orange', linewidth=0.8, alpha=0.6)                    else:                        for g in geom.geoms:                            xs, ys = g.xy                            plt.plot(xs, ys, color='orange', linewidth=0.8, alpha=0.6)            plt.title('Accuratezza orizzontale (IDW vincoli)')            if has_wells or has_sections:                plt.legend(loc='lower left', fontsize=8)            plt.savefig(os.path.join(output_dir, 'horizontal_accuracy_idw.png'), dpi=300, bbox_inches='tight')            plt.close(fig_w)            print("Accuratezza orizzontale calcolata e salvata (CSV + PNG).")        else:            print("Accuratezza orizzontale non calcolata: mancano dati di pozzi/sezioni.")    if has_vertices or has_wells or has_sections:        try:            fig = visualize_data(vertices, triangles, wells_shp, sections_shp, apply_smoothing=False,                                 smoothing_iterations=3, smoothing_factor=0.2, crs='EPSG:6708',                                 output_filename='model_dataset.png')            print("Visualizzazione completata con successo.")        except Exception as e:            print(f"Errore durante la visualizzazione: {e}")            import traceback            traceback.print_exc()    else:        print("Non ci sono dati da visualizzare. Verifica che i file siano presenti nella cartella di lavoro.")    print("Analisi completata.")    return {        'vertices': vertices,        'triangles': triangles,        'wells': wells_shp,        'sections': sections_shp,        'grid_points': grid_points,        'horizontal_weights': grid_weights    }if __name__ == "__main__":    data = main()